<a href="https://colab.research.google.com/github/waghvaishnav/Deep-Learning-PlayGround-Hub/blob/main/Model_Optimization_HyperParameterTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,TensorDataset
import optuna
import torch.nn.functional as f
from sklearn.preprocessing import StandardScaler

In [39]:
# make the synthetic dataset
x,y = make_classification(n_samples=1000,n_features=20,n_classes=2,random_state=42)
x_train,x_val,y_train,y_val = train_test_split(x,y,test_size=0.2,random_state=42)

In [40]:
x_train.shape

(800, 20)

In [41]:
x_val.shape

(200, 20)

In [51]:
# converting this data into the pytorch tensor
x_train,x_val = torch.tensor(x_train,dtype=torch.float32),torch.tensor(x_val,dtype=torch.float32)
y_train,y_val = torch.tensor(y_train,dtype=torch.long),torch.tensor(y_val,dtype=torch.long)

/tmp/ipykernel_16910/139083724.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_train,x_val = torch.tensor(x_train,dtype=torch.float32),torch.tensor(x_val,dtype=torch.float32)
/tmp/ipykernel_16910/139083724.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train,y_val = torch.tensor(y_train,dtype=torch.long),torch.tensor(y_val,dtype=torch.long)


In [52]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)  # Output layer for binary classification
        )

    def forward(self, x):
        return self.network(x)

In [55]:
# creating a optuna function for hyperparametertuning

def objective(trial):

  learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-1)
  hidden_dim = trial.suggest_int("hidden_dim",16,128)

  model = SimpleNN(input_dim=20,hidden_dim=hidden_dim)
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(),lr=learning_rate)

  # model training
  batch_size = 32
  epochs = 20

  train_loader = DataLoader(TensorDataset(x_train,y_train),batch_size=batch_size,shuffle=True)
  test_loader = DataLoader(TensorDataset(x_val,y_val),batch_size=batch_size)

  for epoch in range(epochs):
    model.train()
    for batch_x,batch_y in train_loader:
      # forward pass
      optimizer.zero_grad()
      output = model(batch_x)
      loss = criterion(output,batch_y)
      # backpropagation
      loss.backward()
      # weight updation
      optimizer.step()

  # validation
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for batch_x,batch_y in test_loader:
      output_val = model(batch_x)

      _,predicted = torch.max(output_val,1)
      total += batch_y.size(0)
      correct += (predicted == batch_y).sum().item()

  accuracy = correct / total
  return accuracy

study = optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=20)

[I 2026-03-24 18:15:45,609] A new study created in memory with name: no-name-ed1de63a-27e0-4035-ae84-f7c6078875ec
/tmp/ipykernel_16910/1191277656.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-1)
[I 2026-03-24 18:15:46,464] Trial 0 finished with value: 0.825 and parameters: {'learning_rate': 0.0003881357938811903, 'hidden_dim': 87}. Best is trial 0 with value: 0.825.
[I 2026-03-24 18:15:47,606] Trial 1 finished with value: 0.795 and parameters: {'learning_rate': 0.0034853674914524222, 'hidden_dim': 54}. Best is trial 0 with value: 0.825.
[I 2026-03-24 18:15:48,560] Trial 2 finished with value: 0.82 and parameters: {'learning_rate': 0.0018577728108459051, 'hidden_dim': 104}. Best is trial 0 with value: 0.825.
[I 2026-03-24 18:15:49,565] Trial 3 finished with

In [56]:
study.best_params

{'learning_rate': 0.0012095462406331277, 'hidden_dim': 110}

In [58]:
study.best_value

0.845